# Energy Consumption Forecasting with LSTM

## Objectives
Forecast **household power consumption** (daily mean kW) using LSTM.

## RNN Theory
Time-series forecasting feeds a **sequence of past days** into LSTM; the network outputs the next day's consumption. Gates help remember weekly cycles (weekends vs weekdays).

## Data
UCI Individual Household Electric Power Consumption (aggregated daily).


In [ ]:
# Optional: install dependencies (uncomment if needed)
# !pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow requests yfinance

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)


## Business Context
Utilities and smart-home apps forecast load for **grid planning**, pricing, and anomaly detection.


In [ ]:
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
import zipfile, io, requests
r = requests.get(URL, timeout=120)
z = zipfile.ZipFile(io.BytesIO(r.content))
fname = [n for n in z.namelist() if n.endswith(".txt")][0]
with z.open(fname) as f:
    raw = pd.read_csv(f, sep=";", low_memory=False, na_values=["?"])
raw["datetime"] = pd.to_datetime(raw["Date"] + " " + raw["Time"], dayfirst=True)
raw = raw.set_index("datetime").sort_index()
# Daily mean global active power
daily = raw["Global_active_power"].astype(float).resample("D").mean().dropna().to_frame()
print(daily.head())


In [ ]:
daily.plot(figsize=(12,3))
plt.title("Daily mean global active power (kW)")
plt.show()


In [ ]:
values = daily.values.astype(np.float32)
scaler = MinMaxScaler()
scaled = scaler.fit_transform(values).flatten()
lookback = 30

def make_seq(s, lb):
    X, y = [], []
    for i in range(lb, len(s)):
        X.append(s[i-lb:i])
        y.append(s[i])
    return np.array(X), np.array(y)

X, y = make_seq(scaled, lookback)
X = X[..., np.newaxis]
n = len(X)
tr, va = int(0.7*n), int(0.85*n)
X_train, y_train = X[:tr], y[:tr]
X_val, y_val = X[tr:va], y[tr:va]
X_test, y_test = X[va:], y[va:]


In [ ]:
model = models.Sequential([
    layers.LSTM(64, input_shape=(lookback, 1)),
    layers.Dropout(0.2),
    layers.Dense(1),
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()


In [ ]:
energy_cb = [
    callbacks.ModelCheckpoint("lstm_energy_best.keras", save_best_only=True),
    callbacks.EarlyStopping(patience=6, restore_best_weights=True),
]
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=30,
          callbacks=energy_cb, verbose=1, batch_size=32)
pd.DataFrame(history.history).plot()
plt.show()


In [ ]:
pred = scaler.inverse_transform(model.predict(X_test, verbose=0))
actual = scaler.inverse_transform(y_test.reshape(-1,1))
print("RMSE:", np.sqrt(mean_squared_error(actual, pred)))
plt.plot(actual, label="actual")
plt.plot(pred, label="pred")
plt.legend()
plt.title("Energy consumption — test forecast")
plt.show()


In [ ]:
# Inference — next day forecast
w = scaled[-lookback:].reshape(1, lookback, 1)
nxt = scaler.inverse_transform(model.predict(w, verbose=0))[0,0]
print(f"Next-day power forecast (kW): {nxt:.4f}")
model.save("lstm_energy.keras")
import joblib
joblib.dump(scaler, "energy_scaler.pkl")


## Deployment Notes

1. **Serving**: Export with `model.export("saved_model")` for TensorFlow Serving, or wrap `predict` in FastAPI/Flask.
2. **Preprocessing**: Always apply the **same** scaler/encoder fitted on training data (`scaler.pkl`).
3. **Monitoring**: Track input drift, latency, and prediction distribution on live traffic.
4. **Retraining**: Schedule periodic retrain when performance drops below SLA.
5. **Security**: Do not log PII; use HTTPS and auth on inference endpoints.
